# Лучшие параметры embedding по колонкам

Ноутбук читает полный результат перебора `uzal_embedding_results_all_columns.csv` и оставляет по одной лучшей строке на каждую колонку: минимальный `uzal_cost`, соответствующие `tau` и `dimension`.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

In [ ]:
import subprocess

process = subprocess.Popen(
    ["julia", "--project=.", "main.jl"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)

In [ ]:
RESULTS_PATH = Path("uzal_embedding_results_all_columns.csv")
SKIPPED_PATH = Path("uzal_embedding_skipped_columns.csv")
BEST_OUTPUT_PATH = Path("uzal_best_choices_by_column.csv")

In [ ]:
results = pd.read_csv(RESULTS_PATH)
results.head(), results.shape

In [ ]:
required_columns = {"column", "tau", "dimension", "uzal_cost"}
missing_columns = required_columns - set(results.columns)
if missing_columns:
    raise ValueError(f"Missing columns in {RESULTS_PATH}: {sorted(missing_columns)}")

all_result_columns = set(results["column"].dropna())
finite_results = results.dropna(subset=["column", "tau", "dimension", "uzal_cost"]).copy()
finite_results["tau"] = finite_results["tau"].astype(int)
finite_results["dimension"] = finite_results["dimension"].astype(int)

print(f"Rows in full results: {len(results)}")
print(f"Columns in full results: {len(all_result_columns)}")
print(f"Rows with finite uzal_cost: {len(finite_results)}")
print(f"Columns with finite uzal_cost: {finite_results['column'].nunique()}")

In [ ]:
best_idx = finite_results.groupby("column")["uzal_cost"].idxmin()

best_choices = (
    finite_results.loc[best_idx, ["column", "tau", "dimension", "uzal_cost"]]
    .sort_values("uzal_cost")
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)

display(best_choices)

In [ ]:
from pathlib import Path

paths = [
    Path("uzal_columns_without_finite_cost.csv"),
    Path("uzal_embedding_results_all_columns.csv"),
    Path("uzal_embedding_results.csv"),
    Path("uzal_embedding_skipped_columns.csv"),
    Path("uzal_best_choices_by_column.csv")
]
  
for path in paths:
    if path.exists():
        path.unlink()

best_choices.to_csv("uzal_cost_final_result.csv")